# DeltaNet Effective Beta Diagnostics


In [ ]:
from __future__ import annotations

import math
import sys
import warnings
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter, LogLocator, MaxNLocator, NullFormatter
import numpy as np
import seaborn as sns
import torch
import torch.nn.functional as F

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "PFNs").exists():
    repo_root = repo_root.parent
if not (repo_root / "PFNs").exists():
    raise RuntimeError("Could not find repo root containing PFNs/.")

sys.path.insert(0, str(repo_root / "PFNs"))

from fla.layers.delta_net import DeltaNet
from pfns.experiments.model_benchmarks.model_registry import MODEL_FAMILIES
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
from pfns.experiments.model_benchmarks.plotting import (
    GENERALIZATION_REGION_COLOR,
    PRETRAIN_MAX_X,
    PRETRAIN_REGION_COLOR,
    SPLIT_BOUNDARY_COLOR,
    SPLIT_BOUNDARY_LINESTYLE,
    SPLIT_REGION_ALPHA,
    add_pretraining_split_legend,
    apply_pretraining_split_background,
)
from pfns.model.fla_patches import _apply_deltanet_beta_decay

warnings.filterwarnings(
    "ignore",
    message="ShortConvolution is crucial to the performance.*",
    category=UserWarning,
)

sns.set_theme(
    context="paper",
    style="whitegrid",
    font="DejaVu Sans",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 400,
        "savefig.dpi": 400,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    },
)


## Configuration


In [ ]:
MODEL_NAMES = [
    "DeltaNet_Comb_ST_Reference",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform",
    "DeltaNet_with_lr_decay_online_inverse_t0_256",
    "Non_Causal_DeltaNet",
    "Non_Causal_DeltaNet_loguniform_64K",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256",
]
MODEL_LABELS = {
    "DeltaNet_Comb_ST_Reference": "DeltaNet",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform": "DeltaNet (long sequences)",
    "DeltaNet_with_lr_decay_online_inverse_t0_256": "DeltaNet with decay",
    "Non_Causal_DeltaNet": "Final-state readout DeltaNet",
    "Non_Causal_DeltaNet_loguniform_64K": "Final-state readout DeltaNet\n(long sequences)",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256": "Final-state readout DeltaNet\nwith decay",
}

LAYER_INDICES = [1, 6, 11]
TRAIN_CONTEXT_LEN = 128_000
N_TEST = 64
BATCH_SIZE = 1
N_BETA_FORWARDS = 50
NUM_FEATURES = 10

MAX_CURVE_POINTS = 4096
MAX_MATRIX_POINTS = 1024
POSITION_SAMPLING = "log"  # "log" or "linear"
SMOOTH_BETA = True
SMOOTHING_WINDOW = 101
RETENTION_FLOOR = max(1e-6, float(torch.finfo(torch.float32).eps))
RETENTION_BETA_CEILING = 1.0 - 1e-7
MATRIX_AXIS_SCALES = ["log"]

BETA_PRETRAIN_BOUNDARY = PRETRAIN_MAX_X
BETA_FIGSIZE = (7.2, 3.0)
BETA_GRID_PANEL_SIZE = (3.15, 2.35)
BETA_LINEWIDTH = 2.0
SHOW_BETA_CI = True
BETA_CI_LEVEL = 0.95
BETA_CI_Z = 1.96
BETA_CI_ALPHA = 0.12
BETA_SPLIT_REGION_ALPHA = 0.2
BETA_LEGEND_REGION_ALPHA = 0.55
PLOT_OUTPUT_DIR = repo_root / "PFNs" / "graphics"
PLOT_LAYER_GRID = True
SHOW_LAYER_GRID_LEGENDS = False
SAVE_LAYER_GRID_AS_PDF = True
LAYER_GRID_PDF_FILENAME = "deltanet_effective_beta_layer_grid.pdf"
PLOT_INDIVIDUAL_BETA_LAYERS = False


## Collect Effective Betas


In [ ]:
def model_label(model_name: str) -> str:
    return MODEL_LABELS.get(model_name, model_name)


def find_model_config(model_name: str) -> dict:
    for registry in MODEL_FAMILIES.values():
        if model_name in registry:
            return registry[model_name]
    raise KeyError(f"Could not find model {model_name!r} in MODEL_FAMILIES.")


def load_model(model_name: str):
    loaded_models, loaded_configs = load_models_for_benchmark(
        {model_name: model_configs[model_name]},
        device=str(device),
    )
    model = loaded_models[model_name]
    model.eval()
    return model, loaded_configs[model_name]


def effective_beta(model, hook_output: torch.Tensor) -> torch.Tensor:
    beta = hook_output.detach().float().sigmoid()
    backbone = getattr(model, "transformer_layers", None)
    return _apply_deltanet_beta_decay(
        beta,
        mode=getattr(backbone, "deltanet_beta_decay", "none"),
        t0=int(getattr(backbone, "deltanet_beta_decay_t0", 1000)),
        position_dim=beta.ndim - 2,
    ).cpu()


def make_beta_forward_batches(config: dict) -> list[tuple[torch.Tensor, torch.Tensor]]:
    get_batch = config.priors[0].create_get_batch_method()
    batches = []
    for forward_index in range(N_BETA_FORWARDS):
        batch = get_batch(
            batch_size=BATCH_SIZE,
            seq_len=TRAIN_CONTEXT_LEN + N_TEST,
            num_features=NUM_FEATURES,
            single_eval_pos=TRAIN_CONTEXT_LEN,
            n_targets_per_input=1,
        )
        train_x = batch.x[:, :TRAIN_CONTEXT_LEN].to(device)
        train_y = batch.y[:, :TRAIN_CONTEXT_LEN].to(device)
        batches.append((train_x, train_y))
        print(
            f"forward {forward_index + 1}/{N_BETA_FORWARDS}: "
            f"train_x={tuple(train_x.shape)}, train_y={tuple(train_y.shape)}"
        )
    return batches


def collect_effective_betas(
    model_name: str,
    model,
    forward_batches: list[tuple[torch.Tensor, torch.Tensor]],
) -> list[dict]:
    forward_records = []
    current_records: list[dict] = []
    handles = []

    def hook_for(layer_index: int, layer_name: str):
        def hook(_module, _inputs, output):
            current_records.append(
                {
                    "layer_index": layer_index,
                    "layer": layer_name,
                    "beta": effective_beta(model, output),
                }
            )

        return hook

    for layer_index, (layer_name, layer) in enumerate(
        (item for item in model.named_modules() if isinstance(item[1], DeltaNet) and item[1].use_beta)
    ):
        handles.append(layer.b_proj.register_forward_hook(hook_for(layer_index, layer_name)))

    try:
        for forward_index, (train_x, train_y) in enumerate(forward_batches):
            current_records = []
            with torch.no_grad(), torch.autocast(
                device_type=device.type,
                dtype=autocast_dtype,
                enabled=use_autocast,
            ):
                _ = model.incontext_fit(train_x, train_y)
            if not current_records:
                raise RuntimeError(f"No DeltaNet beta tensors recorded for {model_name}.")
            forward_records.append(current_records)
            print(
                f"{model_label(model_name)}: forward {forward_index + 1}/{len(forward_batches)} "
                f"recorded {len(current_records)} DeltaNet beta tensors"
            )
    finally:
        for handle in handles:
            handle.remove()

    n_layers = len(forward_records[0])
    for forward_index, records in enumerate(forward_records):
        if len(records) != n_layers:
            raise RuntimeError(
                f"Forward {forward_index} recorded {len(records)} layers, expected {n_layers}."
            )

    records = []
    for layer_position in range(n_layers):
        first = forward_records[0][layer_position]
        layer_betas = [records_for_forward[layer_position]["beta"] for records_for_forward in forward_records]
        records.append(
            {
                "layer_index": first["layer_index"],
                "layer": first["layer"],
                "beta": torch.stack(layer_betas, dim=0),
                "n_forwards": len(layer_betas),
            }
        )

    print(
        f"{model_label(model_name)}: aggregated {len(records)} layers across "
        f"{len(forward_batches)} forwards"
    )
    return records


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autocast_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
use_autocast = device.type == "cuda"
model_configs = {model_name: find_model_config(model_name) for model_name in MODEL_NAMES}
print(f"device={device}, use_autocast={use_autocast}, autocast_dtype={autocast_dtype}")

base_model, base_config = load_model(MODEL_NAMES[0])
forward_batches = make_beta_forward_batches(base_config)

beta_records_by_model = {}
for model_name in MODEL_NAMES:
    model = base_model if model_name == MODEL_NAMES[0] else load_model(model_name)[0]
    beta_records_by_model[model_name] = collect_effective_betas(model_name, model, forward_batches)
    if model_name != MODEL_NAMES[0]:
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

for model_name, records in beta_records_by_model.items():
    if max(LAYER_INDICES) >= len(records):
        raise IndexError(f"{model_name} recorded only {len(records)} DeltaNet layers; requested {LAYER_INDICES}.")


## Plot Helpers

In [ ]:
MODEL_COLORS = {
    "Non_Causal_DeltaNet": "#3182BD",
    "Non_Causal_DeltaNet_loguniform_64K": "#08519C",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256": "#3182BD",
    "DeltaNet_Comb_ST_Reference": "#B2182B",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform": "#67000D",
    "DeltaNet_with_lr_decay_online_inverse_t0_256": "#E34A33",
}
MODEL_LINESTYLES = {model_name: "-" for model_name in MODEL_COLORS}
MODEL_LINESTYLES.update({
    "DeltaNet_with_lr_decay_online_inverse_t0_256": "--",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256": "--",
})
BETA_FLOOR = torch.finfo(torch.float32).tiny


def beta_forward_means(record: dict) -> torch.Tensor:
    beta = record["beta"]
    if beta.ndim < 3:
        beta = beta.unsqueeze(0)
    seq_dim = beta.ndim - 2
    seq_len = beta.shape[seq_dim]
    per_forward = beta.movedim(seq_dim, 1).reshape(beta.shape[0], seq_len, -1)
    return per_forward.mean(dim=2).clamp_min(BETA_FLOOR)


def mean_beta(record: dict) -> torch.Tensor:
    return beta_forward_means(record).mean(dim=0).clamp_min(BETA_FLOOR)


def mean_cumulative_retention(record: dict, *, reverse: bool = False) -> torch.Tensor:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    log_retention = torch.log1p(-beta).double()
    if reverse:
        log_retention = log_retention.flip(0).cumsum(dim=0).flip(0) - log_retention
    else:
        log_retention = log_retention.cumsum(dim=0) - log_retention[0]
    return torch.exp(log_retention).clamp_min(RETENTION_FLOOR)


def sample_positions(n_positions: int, max_points: int, sampling: str | None = None) -> torch.Tensor:
    sampling = POSITION_SAMPLING if sampling is None else sampling
    if n_positions <= max_points:
        return torch.arange(n_positions)
    if sampling == "linear":
        return torch.linspace(0, n_positions - 1, max_points).round().long().unique()
    if sampling != "log":
        raise ValueError(f"Unknown position sampling mode {sampling!r}.")

    idx = torch.logspace(0, torch.log10(torch.tensor(float(n_positions))), max_points * 2)
    idx = (idx.round().long() - 1).clamp(0, n_positions - 1).unique()
    if idx.numel() > max_points:
        idx = idx[torch.linspace(0, idx.numel() - 1, max_points).round().long()]
    return idx


def smooth(values: torch.Tensor, window: int = SMOOTHING_WINDOW) -> torch.Tensor:
    values = values.clamp_min(BETA_FLOOR)
    if not SMOOTH_BETA or window <= 1 or values.numel() < 3:
        return values
    window = min(int(window) | 1, values.numel() if values.numel() % 2 else values.numel() - 1)
    log_values = values.log10().view(1, 1, -1)
    kernel = torch.ones(1, 1, window, dtype=log_values.dtype) / float(window)
    return torch.pow(10.0, F.conv1d(F.pad(log_values, (window // 2, window // 2), mode="replicate"), kernel).flatten())


def beta_curve_stats(
    model_name: str,
    layer_index: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    forward_means = beta_forward_means(beta_records_by_model[model_name][layer_index])
    idx = sample_positions(forward_means.shape[1], MAX_CURVE_POINTS)
    curves = torch.stack([smooth(values[idx]) for values in forward_means])
    mean = curves.mean(dim=0).clamp_min(BETA_FLOOR)
    if curves.shape[0] > 1:
        ci = BETA_CI_Z * curves.std(dim=0, unbiased=True) / math.sqrt(curves.shape[0])
    else:
        ci = torch.zeros_like(mean)
    lower = (mean - ci).clamp_min(0.0)
    upper = mean + ci
    return idx + 1, mean, lower, upper


def beta_curve(model_name: str, layer_index: int) -> tuple[torch.Tensor, torch.Tensor]:
    x, mean, _lower, _upper = beta_curve_stats(model_name, layer_index)
    return x, mean


def format_sequence_position(value, _position=None) -> str:
    if value <= 0 or not np.isfinite(value):
        return ""
    value = int(round(float(value)))
    if value >= 1000 and value % 1000 == 0:
        return f"{value // 1000}k"
    if value >= 1000:
        return f"{value / 1000:g}k"
    return str(value)


def style_beta_axis(ax, *, max_position: float, title: str | None = None, ylabel: bool = True, ylim=None) -> None:
    ax.set_xscale("log")
    ax.set_xlim(1.0, max_position)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.tick_params(axis="both", which="major", labelsize=8, pad=2)
    ax.set_xlabel(f"Sequence position ({POSITION_SAMPLING} sampled)", fontsize=9)
    if ylabel:
        ax.set_ylabel(r"Effective write rate, $\beta_t$", fontsize=9)
    if title:
        ax.set_title(title, pad=5, fontsize=10)
    ax.grid(True, which="major", color="#d9d9d9", linewidth=0.75, alpha=0.7)
    ax.grid(True, which="minor", axis="x", color="#e6e6e6", linewidth=0.45, alpha=0.45)
    apply_pretraining_split_background(
        ax,
        boundary=BETA_PRETRAIN_BOUNDARY,
        region_alpha=BETA_SPLIT_REGION_ALPHA,
    )


def add_beta_legends(fig, ax, *, model_loc="center left", model_bbox=(1.02, 0.5)) -> None:
    handles, labels = ax.get_legend_handles_labels()
    add_pretraining_split_legend(ax, boundary=BETA_PRETRAIN_BOUNDARY)
    ax.legend(
        handles,
        labels,
        loc=model_loc,
        bbox_to_anchor=model_bbox,
        frameon=False,
        fontsize=8,
        borderaxespad=0.0,
        labelspacing=0.45,
        handlelength=2.5,
    )
    fig.subplots_adjust(right=0.74)


def position_edges(positions: torch.Tensor, scale: str) -> torch.Tensor:
    x = positions.float() + 1.0
    if scale == "log":
        x = x.log()
    elif scale != "linear":
        raise ValueError(f"Unknown axis scale {scale!r}.")

    if x.numel() == 1:
        edges = torch.tensor([float(x[0] - 0.5), float(x[0] + 0.5)])
    else:
        mid = (x[:-1] + x[1:]) / 2.0
        edges = torch.cat([(x[0] - (mid[0] - x[0]))[None], mid, (x[-1] + (x[-1] - mid[-1]))[None]])
    return edges.exp().clamp_min(1e-6) if scale == "log" else edges.clamp_min(0.5)


def retention_matrix(record: dict) -> tuple[torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    positions = sample_positions(beta.numel(), MAX_MATRIX_POINTS)
    log_cum = torch.log1p(-beta).cumsum(dim=0)[positions]
    matrix = torch.tril(torch.exp(log_cum[:, None] - log_cum[None, :]))
    return positions, matrix.clamp_min(RETENTION_FLOOR)


def combined_zoom_retention_matrix(record: dict) -> tuple[int, torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    n_positions = beta.numel()
    distances = sample_positions(n_positions, MAX_MATRIX_POINTS)
    positions = n_positions - 1 - distances
    log_cum = torch.log1p(-beta).cumsum(dim=0)

    target_grid = positions[:, None]
    source_grid = positions[None, :]
    valid = target_grid >= source_grid
    matrix = torch.exp(log_cum[target_grid] - log_cum[source_grid])
    matrix = torch.where(valid, matrix, torch.full_like(matrix, RETENTION_FLOOR))
    return n_positions, distances, matrix.clamp_min(RETENTION_FLOOR)


def retention_by_target_lag(record: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    n_positions = beta.numel()
    target_positions = sample_positions(n_positions, MAX_MATRIX_POINTS)
    lags = sample_positions(n_positions, MAX_MATRIX_POINTS)
    log_cum = torch.log1p(-beta).cumsum(dim=0)

    source_positions = target_positions[:, None] - lags[None, :]
    valid = source_positions >= 0
    source_positions = source_positions.clamp_min(0).long()
    matrix = torch.exp(log_cum[target_positions, None] - log_cum[source_positions])
    matrix = torch.where(valid, matrix, torch.full_like(matrix, RETENTION_FLOOR))
    return target_positions, lags, matrix.clamp_min(RETENTION_FLOOR)


def final_state_retention_by_lag(record: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    n_positions = beta.numel()
    target_distances = sample_positions(n_positions, MAX_MATRIX_POINTS)
    target_positions = n_positions - 1 - target_distances
    lags = sample_positions(n_positions, MAX_MATRIX_POINTS)
    log_cum = torch.log1p(-beta).cumsum(dim=0)

    source_positions = target_positions[:, None] - lags[None, :]
    valid = source_positions >= 0
    source_positions = source_positions.clamp_min(0).long()
    matrix = torch.exp(log_cum[target_positions, None] - log_cum[source_positions])
    matrix = torch.where(valid, matrix, torch.full_like(matrix, RETENTION_FLOOR))
    return target_distances, lags, matrix.clamp_min(RETENTION_FLOOR)


## Plots


In [ ]:
def beta_plot_label() -> str:
    return "smoothed mean effective beta" if SMOOTH_BETA else "mean effective beta"


def plot_beta_curves(layer_index: int):
    fig, ax = plt.subplots(figsize=BETA_FIGSIZE, constrained_layout=False)
    ymax = 0.0
    max_position = 1.0
    for model_name in MODEL_NAMES:
        x, y, lower, upper = beta_curve_stats(model_name, layer_index)
        ymax = max(ymax, float(upper.max()))
        max_position = max(max_position, float(x[-1]))
        if SHOW_BETA_CI:
            ax.fill_between(
                x.numpy(),
                lower.numpy(),
                upper.numpy(),
                color=MODEL_COLORS[model_name],
                alpha=BETA_CI_ALPHA,
                linewidth=0.0,
                zorder=2,
            )
        ax.plot(
            x.numpy(),
            y.numpy(),
            label=model_label(model_name),
            color=MODEL_COLORS[model_name],
            linestyle=MODEL_LINESTYLES.get(model_name, "-"),
            linewidth=BETA_LINEWIDTH,
            solid_capstyle="round",
            zorder=3,
        )

    style_beta_axis(
        ax,
        max_position=max_position,
        title=f"Layer {layer_index + 1}: {beta_plot_label()}",
        ylim=(0.0, ymax * 1.08),
    )
    add_beta_legends(fig, ax)
    fig.tight_layout()
    plt.show()
    return fig, ax


def plot_beta_layer_grid(
    layer_indices: list[int] | tuple[int, ...] = tuple(LAYER_INDICES),
    *,
    sharey: bool = False,
    show_legend: bool | None = None,
    save_pdf: bool | None = None,
    save_filename: str | Path | None = None,
):
    if show_legend is None:
        show_legend = SHOW_LAYER_GRID_LEGENDS
    if save_pdf is None:
        save_pdf = SAVE_LAYER_GRID_AS_PDF
    if save_filename is None:
        save_filename = LAYER_GRID_PDF_FILENAME

    layer_indices = list(layer_indices)
    ncols = min(3, len(layer_indices))
    nrows = math.ceil(len(layer_indices) / ncols)
    figsize = (BETA_GRID_PANEL_SIZE[0] * ncols, BETA_GRID_PANEL_SIZE[1] * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True, sharey=sharey, squeeze=False)
    flat_axes = axes.ravel()

    all_ymax = 0.0
    all_xmax = 1.0
    curves_by_layer = {}
    for layer_index in layer_indices:
        curves = {}
        for model_name in MODEL_NAMES:
            x, y, lower, upper = beta_curve_stats(model_name, layer_index)
            curves[model_name] = (x, y, lower, upper)
            all_ymax = max(all_ymax, float(upper.max()))
            all_xmax = max(all_xmax, float(x[-1]))
        curves_by_layer[layer_index] = curves

    for panel_index, (ax, layer_index) in enumerate(zip(flat_axes, layer_indices)):
        for model_name in MODEL_NAMES:
            x, y, lower, upper = curves_by_layer[layer_index][model_name]
            if SHOW_BETA_CI:
                ax.fill_between(
                    x.numpy(),
                    lower.numpy(),
                    upper.numpy(),
                    color=MODEL_COLORS[model_name],
                    alpha=BETA_CI_ALPHA,
                    linewidth=0.0,
                    zorder=2,
                )
            ax.plot(
                x.numpy(),
                y.numpy(),
                label=model_label(model_name) if panel_index == 0 else None,
                color=MODEL_COLORS[model_name],
                linestyle=MODEL_LINESTYLES.get(model_name, "-"),
                linewidth=1.75,
                solid_capstyle="round",
                zorder=3,
            )
        panel_ylim = (0.0, all_ymax * 1.08) if sharey else None
        style_beta_axis(
            ax,
            max_position=all_xmax,
            title=f"Layer {layer_index + 1}",
            ylabel=panel_index % ncols == 0,
            ylim=panel_ylim,
        )
        if panel_index // ncols < nrows - 1:
            ax.set_xlabel("")

    layout_rect = (0.0, 0.0, 1.0, 1.0)
    unused_axes = list(flat_axes[len(layer_indices):])
    if not show_legend:
        for ax in unused_axes:
            ax.remove()
    elif unused_axes:
        handles, labels = flat_axes[0].get_legend_handles_labels()
        boundary_label = format_sequence_position(BETA_PRETRAIN_BOUNDARY)
        range_handles = [
            mpatches.Patch(
                facecolor=PRETRAIN_REGION_COLOR,
                alpha=BETA_LEGEND_REGION_ALPHA,
                edgecolor="none",
                label=f"Pre-training range (<={boundary_label})",
            ),
            mpatches.Patch(
                facecolor=GENERALIZATION_REGION_COLOR,
                alpha=BETA_LEGEND_REGION_ALPHA,
                edgecolor="none",
                label=f"Generalization range (>{boundary_label})",
            ),
            mlines.Line2D(
                [],
                [],
                color=SPLIT_BOUNDARY_COLOR,
                linestyle=SPLIT_BOUNDARY_LINESTYLE,
                linewidth=1.5,
                label=f"Pre-training limit ({boundary_label})",
            ),
        ]

        legend_ax = unused_axes[0]
        legend_ax.axis("off")
        legend_ax.text(
            0.0,
            0.98,
            "Models",
            transform=legend_ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            fontweight="semibold",
        )
        model_legend = legend_ax.legend(
            handles,
            labels,
            loc="upper left",
            bbox_to_anchor=(0.0, 0.87),
            ncol=1,
            frameon=False,
            fontsize=7.8,
            handlelength=2.8,
            labelspacing=0.45,
            borderaxespad=0.0,
        )
        legend_ax.add_artist(model_legend)
        legend_ax.text(
            0.0,
            0.31,
            "Ranges",
            transform=legend_ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            fontweight="semibold",
        )
        legend_ax.legend(
            handles=range_handles,
            loc="upper left",
            bbox_to_anchor=(0.0, 0.20),
            frameon=False,
            fontsize=7.8,
            labelspacing=0.32,
            handlelength=2.4,
            handletextpad=0.55,
            borderaxespad=0.0,
        )
        for ax in unused_axes[1:]:
            ax.remove()
    else:
        handles, labels = flat_axes[0].get_legend_handles_labels()
        add_pretraining_split_legend(flat_axes[0], boundary=BETA_PRETRAIN_BOUNDARY)
        fig.legend(
            handles,
            labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.01),
            ncol=min(3, len(labels)),
            frameon=False,
            fontsize=8,
            handlelength=2.5,
        )
        layout_rect = (0.0, 0.08, 1.0, 1.0)

    fig.tight_layout(rect=layout_rect, h_pad=1.6, w_pad=1.2)
    if save_pdf:
        output_path = PLOT_OUTPUT_DIR / save_filename
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    return fig, axes


def plot_beta_layers_for_model(model_name: str, layer_indices: list[int] | tuple[int, ...] = tuple(LAYER_INDICES)):
    fig, ax = plt.subplots(figsize=BETA_FIGSIZE, constrained_layout=False)
    layer_indices = list(layer_indices)
    colors = sns.color_palette("viridis", n_colors=len(layer_indices)).as_hex()
    ymax = 0.0
    max_position = 1.0
    for color, layer_index in zip(colors, layer_indices):
        x, y = beta_curve(model_name, layer_index)
        ymax = max(ymax, float(y.max()))
        max_position = max(max_position, float(x[-1]))
        ax.plot(
            x.numpy(),
            y.numpy(),
            label=f"Layer {layer_index}",
            color=color,
            linewidth=BETA_LINEWIDTH,
            solid_capstyle="round",
            zorder=3,
        )

    style_beta_axis(
        ax,
        max_position=max_position,
        title=f"{model_label(model_name)}: {beta_plot_label()} by layer",
        ylim=(0.0, ymax * 1.08),
    )
    add_beta_legends(fig, ax)
    fig.tight_layout()
    plt.show()
    return fig, ax


def plot_beta_heatmap(model_name: str):
    records = beta_records_by_model[model_name]
    idx = sample_positions(mean_beta(records[0]).numel(), MAX_CURVE_POINTS)
    values = torch.stack([smooth(mean_beta(record)[idx]) for record in records])
    x_edges = position_edges(idx, "log")
    y_edges = torch.arange(len(records) + 1, dtype=torch.float32) - 0.5

    fig, ax = plt.subplots(figsize=(7.2, max(2.6, 0.28 * len(records))))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        values.numpy(),
        shading="auto",
        cmap="viridis",
        vmin=0.0,
        vmax=float(values.max() * 1.05),
        rasterized=True,
    )
    ax.set_xscale("log")
    ax.set_xlim(1.0, float(x_edges[-1]))
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.set_yticks(range(len(records)))
    ax.set_yticklabels(["Layer {}".format(record["layer_index"]) for record in records])
    ax.set_xlabel(f"Sequence position ({POSITION_SAMPLING} sampled)")
    ax.set_ylabel("DeltaNet layer")
    ax.set_title(f"{model_label(model_name)}: {beta_plot_label()} by layer")
    apply_pretraining_split_background(ax, boundary=BETA_PRETRAIN_BOUNDARY, region_alpha=0.18)
    add_pretraining_split_legend(ax, boundary=BETA_PRETRAIN_BOUNDARY)
    fig.colorbar(image, ax=ax, label="Mean effective beta", pad=0.02)
    fig.tight_layout()
    plt.show()
    return fig, ax


def plot_cumulative_retention_heatmap(model_name: str, *, reverse: bool = False):
    records = beta_records_by_model[model_name]
    values = torch.stack([mean_cumulative_retention(record, reverse=reverse) for record in records])
    n_positions = values.shape[1]

    if reverse:
        x_positions = sample_positions(n_positions, MAX_CURVE_POINTS)
        idx = n_positions - 1 - x_positions
        values = values[:, idx]
        xlabel = f"Positions before final token ({POSITION_SAMPLING} sampled; final=1)"
    else:
        idx = sample_positions(n_positions, MAX_CURVE_POINTS)
        x_positions = idx
        values = values[:, idx]
        xlabel = f"Sequence position ({POSITION_SAMPLING} sampled)"

    x_edges = position_edges(x_positions, "log")
    y_edges = torch.arange(len(records) + 1, dtype=torch.float32) - 0.5

    fig, ax = plt.subplots(figsize=(7.2, max(2.6, 0.28 * len(records))))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        values.numpy(),
        shading="auto",
        cmap="magma",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
        rasterized=True,
    )
    ax.set_xscale("log")
    ax.set_xlim(1.0, float(x_edges[-1]))
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.set_yticks(range(len(records)))
    ax.set_yticklabels(["Layer {}".format(record["layer_index"]) for record in records])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("DeltaNet layer")
    direction = "reverse cumulative retention" if reverse else "cumulative retention"
    ax.set_title(f"{model_label(model_name)}: {direction} by layer")
    if not reverse:
        apply_pretraining_split_background(ax, boundary=BETA_PRETRAIN_BOUNDARY, region_alpha=0.18)
        add_pretraining_split_legend(ax, boundary=BETA_PRETRAIN_BOUNDARY)
    fig.colorbar(image, ax=ax, label="Mean cumprod 1 - beta", pad=0.02)
    fig.tight_layout()
    plt.show()
    return fig, ax


def plot_retention_by_lag(model_name: str, layer_index: int):
    target_positions, lags, matrix = retention_by_target_lag(beta_records_by_model[model_name][layer_index])
    x_edges = position_edges(target_positions, "log")
    y_edges = position_edges(lags, "log")

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        matrix.T.numpy(),
        shading="auto",
        cmap="magma",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
        rasterized=True,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1.0, float(x_edges[-1]))
    ax.set_ylim(1.0, float(y_edges[-1]))
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.yaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.set_xlabel("Target sequence position")
    ax.set_ylabel("Lag = target - source")
    ax.set_title(f"{model_label(model_name)}: layer {layer_index} retention horizon by target")
    apply_pretraining_split_background(ax, boundary=BETA_PRETRAIN_BOUNDARY, region_alpha=0.18)
    add_pretraining_split_legend(ax, boundary=BETA_PRETRAIN_BOUNDARY)
    fig.colorbar(image, ax=ax, label="Retention from source to target", pad=0.02)
    fig.tight_layout()
    plt.show()
    return fig, ax


def plot_final_state_retention_by_lag(model_name: str, layer_index: int):
    target_distances, lags, matrix = final_state_retention_by_lag(beta_records_by_model[model_name][layer_index])
    x_edges = position_edges(target_distances, "log")
    y_edges = position_edges(lags, "log")

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        matrix.T.numpy(),
        shading="auto",
        cmap="magma",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
        rasterized=True,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1.0, float(x_edges[-1]))
    ax.set_ylim(1.0, float(y_edges[-1]))
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.yaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.set_xlabel(f"Target distance before final token ({POSITION_SAMPLING} sampled; final=1)")
    ax.set_ylabel("Lag = target - source")
    ax.set_title(f"{model_label(model_name)}: layer {layer_index} final-state retention horizon")
    fig.colorbar(image, ax=ax, label="Retention from source to target", pad=0.02)
    fig.tight_layout()
    plt.show()
    return fig, ax


def plot_retention_matrix(model_name: str, layer_index: int, scale: str, *, reverse: bool = False):
    record = beta_records_by_model[model_name][layer_index]
    if reverse:
        n_positions, distances, matrix = combined_zoom_retention_matrix(record)
        x_edges = position_edges(distances, scale)
        y_edges = x_edges
    else:
        positions, matrix = retention_matrix(record)
        x_edges = position_edges(positions, scale)
        y_edges = x_edges

    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        matrix.numpy(),
        shading="auto",
        cmap="magma",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
        rasterized=True,
    )
    if scale == "log":
        ax.set_xscale("log")
        ax.set_yscale("log")
        if reverse:
            ax.set_xlim(float(x_edges[-1]), 1.0)
            ax.set_ylim(1.0, float(y_edges[-1]))
            ticks = torch.tensor([1, 10, 100, 1_000, 10_000, 100_000, n_positions], dtype=torch.float32)
            ticks = ticks[(ticks >= 1) & (ticks <= n_positions)].unique()
            tick_labels = [f"{int(n_positions - tick.item() + 1):,}" for tick in ticks]
            ax.set_xticks(ticks.tolist())
            ax.set_yticks(ticks.tolist())
            ax.set_xticklabels(tick_labels, rotation=30, ha="right")
            ax.set_yticklabels(tick_labels)
        else:
            ax.set_xlim(1.0, float(x_edges[-1]))
            ax.set_ylim(float(y_edges[-1]), 1.0)
            ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_position))
            ax.yaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    else:
        ax.invert_yaxis()

    ax.set_xlabel("Source sequence position" + (" (reverse-log zoom near end)" if reverse else ""))
    ax.set_ylabel("Target sequence position" + (" (reverse-log zoom near end)" if reverse else ""))
    direction = "retention matrix, reverse-log end zoom" if reverse else "retention matrix"
    ax.set_title(f"{model_label(model_name)}: layer {layer_index} {direction} ({scale}-{scale})")
    if not reverse:
        apply_pretraining_split_background(ax, boundary=BETA_PRETRAIN_BOUNDARY, region_alpha=0.18)
        add_pretraining_split_legend(ax, boundary=BETA_PRETRAIN_BOUNDARY)
    fig.colorbar(image, ax=ax, label="prod(1 - beta)", pad=0.02)
    fig.tight_layout()
    plt.show()
    return fig, ax


## Render


In [ ]:
if PLOT_LAYER_GRID:
    plot_beta_layer_grid(LAYER_INDICES)

# if PLOT_INDIVIDUAL_BETA_LAYERS:
#     for layer_index in LAYER_INDICES:
#         plot_beta_curves(layer_index)

# Overlay layers for a single model when you want one model-specific layer comparison.
# plot_beta_layers_for_model("Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256", LAYER_INDICES)

# for model_name in MODEL_NAMES:
#     plot_beta_heatmap(model_name)

# for model_name in MODEL_NAMES:
#     plot_cumulative_retention_heatmap(model_name, reverse=False)
#     plot_cumulative_retention_heatmap(model_name, reverse=True)

# for model_name in MODEL_NAMES:
#     for layer_index in LAYER_INDICES:
#         for scale in MATRIX_AXIS_SCALES:
#             plot_retention_matrix(model_name, layer_index, scale, reverse=False)
#             plot_retention_matrix(model_name, layer_index, scale, reverse=True)

# for model_name in MODEL_NAMES:
#     for layer_index in LAYER_INDICES:
#         plot_retention_by_lag(model_name, layer_index)

# for model_name in MODEL_NAMES:
#     for layer_index in LAYER_INDICES:
#         plot_final_state_retention_by_lag(model_name, layer_index)


In [ ]:
import numpy as np
from matplotlib.ticker import FuncFormatter, MaxNLocator


def format_sequence_tick(value, _position=None):
    value = int(round(float(value)))
    if abs(value) >= 1000:
        return f"{value / 1000:g}k"
    return str(value)


def plot_banded(M, scale=1.0, log=False, ax=None, num_ticks=10, axis_unit=1.0, **kwargs):
    M = np.asarray(M, dtype=float)
    if M.ndim != 2 or M.shape[0] != M.shape[1]:
        raise ValueError("M must be a square 2D matrix.")
    if log and scale <= 0:
        raise ValueError("scale must be positive when log=True.")
    if axis_unit <= 0:
        raise ValueError("axis_unit must be positive.")

    N = M.shape[0]
    to_y = lambda K: N * np.log1p(K * scale / N) / np.log1p(scale) if log else K * scale
    to_k = lambda y: N * np.expm1(y * np.log1p(scale) / N) / scale if log else y / scale

    arange = np.arange(N + 1, dtype=float)
    I, J = np.meshgrid(arange, arange, indexing="ij")

    K = np.abs(I - J)
    Z = np.maximum(N - K, 1.0)
    K = np.where(I > J, to_y(K), K)
    base = np.minimum(I, J) * (N - K) / Z
    I, J = base + K * (I > J), base + K * (I < J)

    if ax is None:
        _, ax = plt.subplots()
    kwargs.setdefault("shading", "flat")
    kwargs.setdefault("rasterized", True)
    kwargs.setdefault("cmap", "viridis")
    mesh = ax.pcolormesh(J * axis_unit, I * axis_unit, M, **kwargs)

    ax.set_aspect("equal")
    ax.set_xlim(0, N * axis_unit)
    ax.set_ylim(N * axis_unit, 0)

    ax.set_xlabel("source position", labelpad=8)
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    ax.tick_params(axis="x", which="major", pad=6)
    ax.spines["top"].set_visible(True)

    ax.set_ylabel("target position", labelpad=10, rotation=270, va="bottom")
    ax.yaxis.set_ticks_position("right")
    ax.yaxis.set_label_position("right")
    ax.tick_params(axis="y", which="major", pad=6)
    ax.spines["right"].set_visible(True)

    ax_l = ax.secondary_yaxis(
        "left",
        functions=(lambda y: to_k(y / axis_unit) * axis_unit, lambda K: to_y(K / axis_unit) * axis_unit),
    )
    ax_l.set_ylabel("lag (target - source)", labelpad=10)
    ax_l.tick_params(axis="y", which="major", pad=5)
    ax_b = ax.secondary_xaxis(
        "bottom",
        functions=(
            lambda x: to_k(N - x / axis_unit) * axis_unit,
            lambda K: (N - to_y(K / axis_unit)) * axis_unit,
        ),
    )
    ax_b.set_xlabel("lag (target - source)", labelpad=8)
    ax_b.tick_params(axis="x", which="major", pad=6)

    if log:
        k_max = N * axis_unit
        hi = int(np.log10(k_max))
        major = 10.0 ** np.arange(hi + 1)
        major = major[major <= k_max]
        major = major[np.append(np.diff(to_y(major / axis_unit) / N) >= 0.05, True)]
        minor = np.array([k * 10.0**e for e in range(-1, hi + 1) for k in range(2, 10)])
        minor = minor[(minor > major[0]) & (minor < major[-1])]
        for axis in (ax_l.yaxis, ax_b.xaxis):
            axis.set_ticks(major, [f"$10^{{{int(np.log10(k))}}}$" for k in major])
            axis.set_ticks(minor, minor=True)

    ax.xaxis.set_major_locator(MaxNLocator(nbins=num_ticks, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=num_ticks, integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_tick))
    ax.yaxis.set_major_formatter(FuncFormatter(format_sequence_tick))
    grid_kwargs = dict(color=plt.rcParams["grid.color"], lw=plt.rcParams["grid.linewidth"])
    for tick in ax.get_xticks():
        idx = int(round(tick / axis_unit))
        if 0 <= idx <= N:
            ax.plot(J[:, idx] * axis_unit, I[:, idx] * axis_unit, **grid_kwargs)
            ax.plot(J[N - idx, :] * axis_unit, I[N - idx, :] * axis_unit, **grid_kwargs)
    return mesh


def banded_retention_matrix(record: dict, max_points: int = MAX_MATRIX_POINTS) -> tuple[torch.Tensor, torch.Tensor, int]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    positions = sample_positions(beta.numel(), max_points, sampling="linear")
    log_cum = torch.log1p(-beta).cumsum(dim=0)[positions]
    matrix = torch.tril(torch.exp(log_cum[:, None] - log_cum[None, :]))
    return positions, matrix.clamp_min(RETENTION_FLOOR), beta.numel()


def plot_banded_retention_matrix(
    model_name: str,
    layer_index: int,
    *,
    scale: float = 1000.0,
    log: bool = True,
    max_points: int = MAX_MATRIX_POINTS,
) -> None:
    positions, matrix, n_positions = banded_retention_matrix(
        beta_records_by_model[model_name][layer_index], max_points=max_points
    )
    axis_unit = n_positions / positions.numel()

    fig, ax = plt.subplots(figsize=(10.5, 8.5))
    mesh = plot_banded(
        matrix.numpy(),
        scale=scale,
        log=log,
        ax=ax,
        num_ticks=8,
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
        axis_unit=axis_unit,
    )
    ax.set_title(
        f"{model_label(model_name)} | layer {layer_index} source-to-target retention"
        f" ({positions.numel():,} samples across {n_positions:,} steps)",
        pad=16,
    )
    fig.colorbar(
        mesh,
        ax=ax,
        label="contribution of source to target",
        fraction=0.04,
        pad=0.12,
    )
    fig.subplots_adjust(left=0.15, right=0.78, top=0.86, bottom=0.16)
    plt.show()


# for model_name in MODEL_NAMES:
#     for layer_index in LAYER_INDICES:
#         plot_banded_retention_matrix(model_name, layer_index)


In [ ]:
from matplotlib.colors import LogNorm, TwoSlopeNorm

FINAL_STATE_CONTRIBUTION_FLOOR = 1e-6
FINAL_STATE_CONTRIBUTION_CMAP = "viridis"
FINAL_STATE_EFFECT_CMAP = "RdBu_r"
FINAL_STATE_CONTRIBUTION_XLABEL = f"Samples before final context sample (final position = {format_sequence_position(TRAIN_CONTEXT_LEN)})"
FINAL_STATE_CONTRIBUTION_COLORBAR_LABEL = "Contribution to final state"
FINAL_STATE_MAX_SAMPLES_BEFORE_FINAL = 10_000
FINAL_STATE_EFFECT_MAX_SAMPLES_BEFORE_FINAL = 10_000
FINAL_STATE_EFFECT_COLORBAR_LABEL = r"$\log_{10}(C_{\mathrm{long}}/C_{\mathrm{standard}})$"
FINAL_STATE_CONTRIBUTION_VMAX = 1.0
FINAL_STATE_EFFECT_LIMIT_QUANTILE = 0.985
FINAL_STATE_EFFECT_MIN_ABS_LIMIT = 0.5
FINAL_STATE_EFFECT_MAX_ABS_LIMIT = 2.0
FINAL_STATE_EFFECT_FLOOR = 1e-6
FINAL_STATE_EFFECT_MASK_FLOOR = 1e-6
PLOT_FINAL_STATE_READOUT_CONTRIBUTION_COMPARISON = False
PLOT_BANDED_RETENTION_COMPARISON = False
SAVE_PLOTS_AS_PDF = True
PLOT_OUTPUT_DIR = repo_root / "PFNs" / "graphics"
BANDED_RETENTION_COMPARISON_LAYER = 9  # zero-indexed; shown as Layer 10

DELTANET_CONTRIBUTION_PAIR = (
    "DeltaNet_Comb_ST_Reference",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform",
)
FINAL_STATE_READOUT_CONTRIBUTION_PAIR = (
    "Non_Causal_DeltaNet",
    "Non_Causal_DeltaNet_loguniform_64K",
)


def raw_final_state_contribution_by_layer(record: dict) -> torch.Tensor:
    beta = beta_forward_means(record).clamp(max=RETENTION_BETA_CEILING)
    log_one_minus_beta = torch.log1p(-beta)
    log_survival_after_source = (
        log_one_minus_beta.flip(dims=(1,)).cumsum(dim=1).flip(dims=(1,))
        - log_one_minus_beta
    )
    contribution = beta * torch.exp(log_survival_after_source)
    return contribution.mean(dim=0).clamp_min(0.0)


def final_state_contribution_by_layer(record: dict) -> torch.Tensor:
    return raw_final_state_contribution_by_layer(record).clamp_min(FINAL_STATE_CONTRIBUTION_FLOOR)


def final_state_contribution_heatmap_values(model_name: str):
    records = beta_records_by_model[model_name]
    n_positions = final_state_contribution_by_layer(records[0]).numel()
    max_offset = min(n_positions, FINAL_STATE_MAX_SAMPLES_BEFORE_FINAL)
    sampled_offsets = sample_positions(max_offset, MAX_CURVE_POINTS)
    source_indices = (n_positions - 1 - sampled_offsets).long()
    values = torch.stack(
        [final_state_contribution_by_layer(record)[source_indices] for record in records]
    )
    x_edges = position_edges(sampled_offsets, "log")
    y_edges = torch.arange(len(records) + 1, dtype=torch.float32) - 0.5
    return sampled_offsets + 1, x_edges, y_edges, values


def _compact_model_title(model_name: str) -> str:
    return " ".join(model_label(model_name).splitlines())


def _save_plot_pdf(fig, filename: str | Path | None) -> Path | None:
    if not SAVE_PLOTS_AS_PDF or filename is None:
        return None
    output_path = PLOT_OUTPUT_DIR / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight")
    return output_path


def _contribution_norm(*value_tensors: torch.Tensor) -> LogNorm:
    return LogNorm(vmin=FINAL_STATE_CONTRIBUTION_FLOOR, vmax=FINAL_STATE_CONTRIBUTION_VMAX)


def _raw_contribution_ratio_values(
    standard_values: torch.Tensor,
    long_values: torch.Tensor,
) -> np.ma.MaskedArray:
    standard = standard_values.clamp_min(FINAL_STATE_EFFECT_FLOOR)
    long = long_values.clamp_min(FINAL_STATE_EFFECT_FLOOR)
    effect = torch.log10(long / standard).numpy()
    inactive = torch.maximum(standard_values, long_values).numpy() < FINAL_STATE_EFFECT_MASK_FLOOR
    return np.ma.masked_where(inactive, effect)


def _effect_norm(effect_values: np.ma.MaskedArray) -> TwoSlopeNorm:
    finite_values = effect_values.compressed()
    finite_values = finite_values[np.isfinite(finite_values)]
    if finite_values.size == 0:
        limit = FINAL_STATE_EFFECT_MIN_ABS_LIMIT
    else:
        limit = float(np.quantile(np.abs(finite_values), FINAL_STATE_EFFECT_LIMIT_QUANTILE))
        limit = min(max(limit, FINAL_STATE_EFFECT_MIN_ABS_LIMIT), FINAL_STATE_EFFECT_MAX_ABS_LIMIT)
    return TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)


def _display_lag_limit(x_edges: torch.Tensor) -> float:
    return min(float(x_edges[-1]), float(FINAL_STATE_MAX_SAMPLES_BEFORE_FINAL))


def _display_effect_lag_limit(x_edges: torch.Tensor) -> float:
    return min(float(x_edges[-1]), float(FINAL_STATE_EFFECT_MAX_SAMPLES_BEFORE_FINAL))


def _style_contribution_axis(ax, *, show_y: bool) -> None:
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_position))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.tick_params(axis="both", which="major", labelsize=7.4, pad=2)
    ax.tick_params(axis="x", which="minor", length=1.8)
    ax.grid(False)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_linewidth(0.6)
    ax.spines["bottom"].set_linewidth(0.6)
    if not show_y:
        ax.tick_params(axis="y", left=False, labelleft=False)


def _draw_contribution_panel(ax, model_name: str, norm: LogNorm, *, panel_label: str, show_y: bool):
    _distances, x_edges, y_edges, values = final_state_contribution_heatmap_values(model_name)
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        values.numpy(),
        shading="auto",
        cmap=FINAL_STATE_CONTRIBUTION_CMAP,
        norm=norm,
        rasterized=True,
    )
    ax.set_xlim(_display_lag_limit(x_edges), 1.0)
    ax.set_title(f"{panel_label} {_compact_model_title(model_name)}", fontsize=8.4, pad=4)
    if show_y:
        ax.set_yticks(range(len(beta_records_by_model[model_name])))
        ax.set_yticklabels(
            ["{}".format(record["layer_index"] + 1) for record in beta_records_by_model[model_name]],
            fontsize=7.4,
        )
        ax.set_ylabel("Layer", fontsize=8.4, labelpad=3)
    _style_contribution_axis(ax, show_y=show_y)
    return image, x_edges, y_edges, values


def _draw_effect_panel(
    ax,
    x_edges: torch.Tensor,
    y_edges: torch.Tensor,
    effect_values: np.ma.MaskedArray,
    effect_norm: TwoSlopeNorm,
):
    effect_cmap = plt.get_cmap(FINAL_STATE_EFFECT_CMAP).copy()
    effect_cmap.set_bad("#f2f2f2")
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        effect_values,
        shading="auto",
        cmap=effect_cmap,
        norm=effect_norm,
        rasterized=True,
    )
    ax.set_xlim(_display_effect_lag_limit(x_edges), 1.0)
    ax.set_facecolor("#f2f2f2")
    ax.set_title("(c) Relative contribution shift", fontsize=8.4, pad=4)
    _style_contribution_axis(ax, show_y=False)
    return image


def plot_contribution_pair_comparison(
    standard_model_name: str,
    long_sequence_model_name: str,
    *,
    figure_title: str | None = None,
    save_filename: str | Path | None = None,
):
    _, _, _, standard_values = final_state_contribution_heatmap_values(standard_model_name)
    _, _, _, long_values = final_state_contribution_heatmap_values(long_sequence_model_name)
    contribution_norm = _contribution_norm(standard_values, long_values)
    effect_values = _raw_contribution_ratio_values(standard_values, long_values)
    effect_norm = _effect_norm(effect_values)

    fig = plt.figure(figsize=(9.3, 2.75), constrained_layout=False)
    grid = fig.add_gridspec(
        nrows=1,
        ncols=7,
        width_ratios=[1.0, 0.035, 1.0, 0.045, 0.12, 1.0, 0.045],
        wspace=0.08,
    )
    ax_standard = fig.add_subplot(grid[0, 0])
    ax_long = fig.add_subplot(grid[0, 2])
    cax_contribution = fig.add_subplot(grid[0, 3])
    ax_effect = fig.add_subplot(grid[0, 5])
    cax_effect = fig.add_subplot(grid[0, 6])

    image_standard, x_edges, y_edges, _ = _draw_contribution_panel(
        ax_standard,
        standard_model_name,
        contribution_norm,
        panel_label="(a)",
        show_y=True,
    )
    _draw_contribution_panel(
        ax_long,
        long_sequence_model_name,
        contribution_norm,
        panel_label="(b)",
        show_y=False,
    )
    effect_image = _draw_effect_panel(ax_effect, x_edges, y_edges, effect_values, effect_norm)

    contribution_cbar = fig.colorbar(
        image_standard,
        cax=cax_contribution,
        ticks=[10.0**power for power in range(-6, 1)],
    )
    contribution_cbar.set_label(FINAL_STATE_CONTRIBUTION_COLORBAR_LABEL, fontsize=7.3, rotation=270, labelpad=8)
    contribution_cbar.ax.tick_params(labelsize=6.9, pad=1.5, length=2.0)

    effect_cbar = fig.colorbar(effect_image, cax=cax_effect)
    effect_cbar.set_label(FINAL_STATE_EFFECT_COLORBAR_LABEL, fontsize=7.3, rotation=270, labelpad=8)
    effect_cbar.ax.tick_params(labelsize=6.9, pad=1.5, length=2.0)

    fig.supxlabel(FINAL_STATE_CONTRIBUTION_XLABEL, fontsize=8.4, y=0.045)
    if figure_title:
        fig.suptitle(figure_title, fontsize=9.5, y=0.985)
        top = 0.84
    else:
        top = 0.91
    fig.subplots_adjust(left=0.065, right=0.975, bottom=0.20, top=top)
    _save_plot_pdf(fig, save_filename)
    plt.show()
    return fig, (ax_standard, ax_long, ax_effect)


def plot_banded_retention_comparison(
    model_names: list[str],
    *,
    layer_index: int = BANDED_RETENTION_COMPARISON_LAYER,
    max_points: int = 768,
    save_filename: str | Path | None = None,
):
    fig, axes = plt.subplots(1, len(model_names), figsize=(7.2, 3.2), sharex=False, sharey=False)
    axes = np.atleast_1d(axes).tolist()
    meshes = []
    for ax, model_name in zip(axes, model_names):
        positions, matrix, n_positions = banded_retention_matrix(
            beta_records_by_model[model_name][layer_index], max_points=max_points
        )
        axis_unit = n_positions / positions.numel()
        mesh = plot_banded(
            matrix.numpy(),
            scale=1000.0,
            log=True,
            ax=ax,
            num_ticks=5,
            norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
            cmap=FINAL_STATE_CONTRIBUTION_CMAP,
            axis_unit=axis_unit,
        )
        meshes.append(mesh)
        ax.set_title(f"{_compact_model_title(model_name)} | Layer {layer_index + 1}", fontsize=8.6, pad=5)
        ax.tick_params(axis="both", which="major", labelsize=7.5, pad=2)
    cbar = fig.colorbar(meshes[-1], ax=axes, pad=0.03, fraction=0.035)
    cbar.set_label("Source-to-target retention proxy", fontsize=8.0)
    cbar.ax.tick_params(labelsize=7.3)
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.20, top=0.82, wspace=0.35)
    _save_plot_pdf(fig, save_filename)
    plt.show()
    return fig, axes


plot_contribution_pair_comparison(
    *DELTANET_CONTRIBUTION_PAIR,
    save_filename="deltanet_final_state_contribution_comparison.pdf",
)

plot_contribution_pair_comparison(
    *FINAL_STATE_READOUT_CONTRIBUTION_PAIR,
    save_filename="final_state_readout_deltanet_contribution_comparison.pdf",
)

if PLOT_BANDED_RETENTION_COMPARISON:
    plot_banded_retention_comparison(
        list(DELTANET_CONTRIBUTION_PAIR),
        save_filename=f"deltanet_banded_retention_layer_{BANDED_RETENTION_COMPARISON_LAYER + 1}.pdf",
    )
